# 臺灣高解析度海岸線與 USGS 地震地圖
RustGIS + Natural Earth 1:10m + USGS FDSN API。Python 3.12+，需網路。由上而下執行。
Natural Earth 的 10m 是 1:10,000,000 比例尺，並非 10 公尺解析度；適合全臺尺度，局部研究可另用 GSHHG full。
[官方 RustGIS Python 文件](https://rustgis.app/python/) · [Natural Earth](https://www.naturalearthdata.com/downloads/10m-physical-vectors/10m-coastline/) · [USGS API](https://earthquake.usgs.gov/fdsnws/event/1/).
本 Notebook 尚未在 Colab 實際執行。

In [ ]:
%pip install -q "rustgis[jupyter]" geopandas matplotlib requests

## 1 設定查詢條件
日期使用 UTC，範例為 2024 年 4 月花蓮地震前後。

In [ ]:
# @title 地震查詢參數
starttime = "2024-04-01" # @param {type:"date"}
endtime = "2024-05-01" # @param {type:"date"}
minmagnitude = 4.0 # @param {type:"number"}
lon_min = 119.0 # @param {type:"number"}
lon_max = 123.5 # @param {type:"number"}
lat_min = 21.5 # @param {type:"number"}
lat_max = 26.0 # @param {type:"number"}

import geopandas as gpd
import pandas as pd
import requests
import matplotlib.pyplot as plt
import numpy as np
import json
from pathlib import Path
from shapely.geometry import box
from IPython.display import Image, display
from datetime import date

assert date.fromisoformat(starttime) < date.fromisoformat(endtime)
assert -180 <= lon_min < lon_max <= 180 and -90 <= lat_min < lat_max <= 90
region = box(lon_min, lat_min, lon_max, lat_max)
out = Path("outputs")
out.mkdir(exist_ok=True)

## 2 下載海岸線並裁切
保留原始頂點，不額外簡化。

In [ ]:
coast_url = "https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/ne_10m_coastline.geojson"
coast = gpd.read_file(coast_url).to_crs("EPSG:4326")
taiwan_coast = gpd.clip(coast, region)
taiwan_coast = taiwan_coast[~taiwan_coast.geometry.is_empty].copy()
assert not taiwan_coast.empty, "範圍內無海岸線"
coast_path = out / "taiwan_coastline.geojson"
taiwan_coast.to_file(coast_path, driver="GeoJSON")
print("海岸線圖徵:", len(taiwan_coast))
ax = taiwan_coast.plot(figsize=(9, 9), color="black", linewidth=0.8)
ax.set(xlim=(lon_min,lon_max), ylim=(lat_min,lat_max))
plt.show()

## 3 USGS 地震目錄
查詢結果可能隨目錄修訂而變動；保留原始 JSON、查詢網址及 UTC 擷取時間。規模類型不一定等同 CWA ML。

In [ ]:
url = "https://earthquake.usgs.gov/fdsnws/event/1/query"
params = dict(format="geojson", starttime=starttime, endtime=endtime,
              minlatitude=lat_min, maxlatitude=lat_max,
              minlongitude=lon_min, maxlongitude=lon_max,
              minmagnitude=minmagnitude, orderby="time-asc", limit=20000)
response = requests.get(url, params=params, timeout=120)
response.raise_for_status()
data = response.json()
if len(data["features"]) >= 20000:
    raise ValueError("已達查詢上限，請縮短時間或分段查詢")
(out / "usgs_raw.json").write_text(json.dumps(data), encoding="utf-8")
(out / "query.json").write_text(json.dumps(dict(url=response.url, retrieved_utc=pd.Timestamp.now(tz="UTC").isoformat()), indent=2))
records = []
for event in data["features"]:
    lon, lat, depth = event["geometry"]["coordinates"]
    p = event["properties"]
    if p["mag"] is None or depth is None:
        continue
    records.append(dict(event_id=event["id"], longitude=lon, latitude=lat,
        depth=depth, magnitude=p["mag"], mag_type=p.get("magType"),
        place=p.get("place"), time=pd.to_datetime(p["time"],unit="ms",utc=True).isoformat()))
df = pd.DataFrame(records)
if df.empty:
    raise ValueError("沒有符合條件且具有規模與深度的地震")
eq = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.longitude, df.latitude), crs="EPSG:4326")
eq_path = out / "usgs_earthquakes.geojson"
eq.to_file(eq_path, driver="GeoJSON")
df.to_csv(out / "usgs_earthquakes.csv", index=False)
print("地震數:", len(eq))
display(df.head())

## 4 RustGIS 繪圖：規模比例符號、深度分級
地圖輸出 2400 × 2400 像素。兩種符號化分別輸出。

In [ ]:
import rustgis
s = rustgis.Session()
s.set_basemap("blank", color="#DCEEF8")
coast_layer = s.add_geojson(str(coast_path), name="Taiwan Coastline")
eq_layer = s.add_geojson(str(eq_path), name="USGS Earthquakes")
assert eq_layer.count == len(eq)
eq_layer.style("proportional", field="magnitude")
s.zoom_to((lon_min,lat_min,lon_max,lat_max))
s.to_png(str(out / "taiwan_earthquakes.png"), width=2400, height=2400)
display(Image(filename=str(out / "taiwan_earthquakes.png")))

In [ ]:
eq_layer.style("graduated", field="depth")
s.to_png(str(out / "taiwan_earthquake_depth.png"), width=2400,height=2400)
display(Image(filename=str(out / "taiwan_earthquake_depth.png")))
s.save(str(out / "Taiwan_Earthquakes.rgis"))

## 5 互動式 RustGIS
需連線至官方 web app。若 widget 無法顯示，仍可使用 PNG 或將 GeoJSON 匯入 https://web.rustgis.app。官方文件指出可嵌入的單一檔案大小有限制（2 MiB）。

In [ ]:
from google.colab import output
output.enable_custom_widget_manager()
m = rustgis.Map(session=s, height="850px")
m

## 6 科學圖件：深度顏色 + 規模大小
Matplotlib 獨立輸出，可在 RustGIS widget 不相容時使用。符號面積是展示編碼，不是地震能量。

In [ ]:
fig, ax = plt.subplots(figsize=(10,11), dpi=150)
ax.set_facecolor("#E8F4FA")
taiwan_coast.plot(ax=ax,color="black",linewidth=0.8)
eq_sorted = eq.sort_values("magnitude")
sizes = 10 * 2 ** (eq_sorted.magnitude - 3)
sc = ax.scatter(eq_sorted.longitude,eq_sorted.latitude,s=sizes,
    c=eq_sorted.depth,cmap="turbo_r",vmin=0,vmax=100,alpha=0.8,
    edgecolors="black",linewidths=0.35)
fig.colorbar(sc,ax=ax,shrink=0.7,extend="both",label="Focal depth (km)")
for mag in [4,5,6,7]:
    ax.scatter([],[],s=10*2**(mag-3),facecolors="none",edgecolors="black",label=f"M {mag}")
ax.legend(title="Magnitude",loc="lower left")
ax.set(xlim=(lon_min,lon_max),ylim=(lat_min,lat_max),
       xlabel="Longitude (°E)",ylabel="Latitude (°N)")
ax.set_aspect(1/np.cos(np.radians((lat_min+lat_max)/2)))
ax.set_title(f"Taiwan Seismicity\n{starttime} – {endtime} (UTC)\nM ≥ {minmagnitude}, N = {len(eq)}")
ax.grid(linestyle="--",alpha=0.3)
fig.text(0.5,0.02,"Earthquakes: USGS | Coastline: Natural Earth 1:10m",ha="center",fontsize=9)
fig.savefig(out / "Taiwan_Seismicity.png",dpi=300,bbox_inches="tight")
fig.savefig(out / "Taiwan_Seismicity.svg",bbox_inches="tight")
plt.show()

## 7 打包下載
包含資料、查詢紀錄、PNG、SVG 及已成功建立的 RustGIS 專案。

In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive("Taiwan_Seismicity","zip",root_dir=out)
files.download(archive)

## 探究練習
比較 M≥4 與 M≥5 的分布；改變時間窗與深度色階；比較震央與海岸線的關係。討論 USGS 目錄完整性、規模類型及地震深度不確定性。不得將震央密度直接解讀為地震危害。